Import

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

Loading

In [2]:
all_Q5107 = pd.read_csv(
    "../data/Q5107 All National General.csv",
    low_memory=False,
    dtype_backend="numpy_nullable"
    )
print(all_Q5107.head)
print(len(all_Q5107)) # Should be 317874

<bound method NDFrame.head of         PROVIDER_ID                       RAW_PAYER_NAME  \
0              1303                             No payer   
1              1303                             No payer   
2             10367  KAISER FOUNDATION HEALTH PLAN, INC.   
3             10367  KAISER FOUNDATION HEALTH PLAN, INC.   
4             10367  KAISER FOUNDATION HEALTH PLAN, INC.   
...             ...                                  ...   
317869           46                           HEALTH NET   
317870           46                    UNITED HEALTHCARE   
317871           46                          BLUE SHIELD   
317872         3702                             No payer   
317873         3702                             No payer   

                                    PROVIDER_NAME MEDICARE_PROVIDER_ID  \
0                              Penn Highlands Elk               391315   
1                              Penn Highlands Elk               391315   
2           Kaiser Permanen

In [3]:

test=all_Q5107[all_Q5107["PROVIDER_NAME"]=="IU Health University Hospital"]

print(test["GROSS_CHARGE"].isna().sum())

360


Drop Outliers (around 50% are outliers)

In [4]:
317874-138362

179512

In [5]:
print(all_Q5107["RATE_IS_OUTLIER"].sum())
all_Q5107_cleaned = all_Q5107[all_Q5107["RATE_IS_OUTLIER"]==False] # Remove outliers
print(len(all_Q5107_cleaned)) # Should be 179512
print(all_Q5107_cleaned.head)
print(len(all_Q5107_cleaned))
print("CLEANED: NA per column:")
print(all_Q5107_cleaned.isna().sum()) # List NA for each column
print("CLEANED: Number unique per column:")
print(all_Q5107_cleaned.nunique())

138362
179512
<bound method NDFrame.head of         PROVIDER_ID                       RAW_PAYER_NAME  \
0              1303                             No payer   
1              1303                             No payer   
2             10367  KAISER FOUNDATION HEALTH PLAN, INC.   
3             10367  KAISER FOUNDATION HEALTH PLAN, INC.   
4             10367  KAISER FOUNDATION HEALTH PLAN, INC.   
...             ...                                  ...   
317869           46                           HEALTH NET   
317870           46                    UNITED HEALTHCARE   
317871           46                          BLUE SHIELD   
317872         3702                             No payer   
317873         3702                             No payer   

                                    PROVIDER_NAME MEDICARE_PROVIDER_ID  \
0                              Penn Highlands Elk               391315   
1                              Penn Highlands Elk               391315   
2           K

Check Ingest Dates

In [6]:
print(all_Q5107_cleaned["LOADED_ON"].value_counts())
print(all_Q5107_cleaned["INGESTED_ON"].value_counts())

LOADED_ON
2026-06-02 20:57:43.740    100537
2026-06-10 05:59:09.756     36210
2026-06-09 05:47:45.893     17949
2026-06-10 10:24:26.355      9666
2026-06-15 05:56:06.716      5106
2026-06-11 21:33:16.661      4387
2026-06-18 01:14:19.529      2031
2026-06-08 03:53:01.778      1855
2026-06-15 20:46:53.553      1671
2026-06-16 16:56:17.553        46
2026-06-04 20:34:18.863        29
2026-06-05 21:34:39.107        25
Name: count, dtype: Int64
INGESTED_ON
2026-05-26 20:18:59.990    1627
2026-05-02 01:17:20.088    1627
2026-05-04 15:08:38.108    1627
2026-05-04 14:30:19.035    1627
2026-05-04 17:33:57.644    1627
                           ... 
2025-10-13 00:54:07.719       1
2025-07-07 12:39:36.288       1
2026-04-20 00:53:46.815       1
2026-04-20 01:38:09.968       1
2026-04-22 08:28:13.374       1
Name: count, Length: 3063, dtype: Int64


Find Providers That Share Location With Different ID

Findings: some parts of the same campus are different "providers" (example: main vs childrens' hospital)

In [7]:
same_address_diff_ID = all_Q5107_cleaned[all_Q5107_cleaned.groupby("STREET_ADDRESS")["PROVIDER_ID"].transform("nunique")>1]

Check For Duplicates (Cleans duplicates with the same "PROVIDER_ID", "PAYER_ID", "PAYER_CLASS_NAME", "NEGOTIATED_DOLLAR","NEGOTIATED_PERCENTAGE","GROSS_CHARGE", "ESTIMATED_ALLOWED_AMOUNT", "LOADED_ON", "INGESTED_ON", "BILLING_CLASS", "SETTING")

In [8]:
identifiers = ["PROVIDER_ID", "PAYER_ID", "PLAN_NAME", "PAYER_CLASS_NAME", "NEGOTIATED_DOLLAR","NEGOTIATED_PERCENTAGE","GROSS_CHARGE", "ESTIMATED_ALLOWED_AMOUNT", "LOADED_ON", "INGESTED_ON", "BILLING_CLASS", "SETTING"]

print("Are there duplicates?")
print(all_Q5107_cleaned.duplicated(subset = identifiers).any())

duplicates = all_Q5107_cleaned[all_Q5107_cleaned.duplicated(subset = identifiers, keep = False)]
print("Number of duplicates (by logic): " + str(len(duplicates)))
duplicates_entire_row = all_Q5107_cleaned[all_Q5107_cleaned.duplicated(keep = False)]
print("Number of duplicates (by entire row): " + str(len(duplicates_entire_row)))

print("Number of original rows: " + str(len(all_Q5107_cleaned)))


all_Q5107_cleaned = all_Q5107_cleaned.drop_duplicates(keep="first",subset=identifiers)
duplicates = all_Q5107_cleaned[all_Q5107_cleaned.duplicated(subset = identifiers, keep = False)]
duplicates_entire_row = all_Q5107_cleaned[all_Q5107_cleaned.duplicated(keep = False)]
print("Number of duplicates (by logic): " + str(len(duplicates)))
print("Number of duplicates (by entire row): " + str(len(duplicates_entire_row)))
print("Number of new rows: " + str(len(all_Q5107_cleaned)))



Are there duplicates?
True
Number of duplicates (by logic): 75812
Number of duplicates (by entire row): 43698
Number of original rows: 179512
Number of duplicates (by logic): 0
Number of duplicates (by entire row): 0
Number of new rows: 128688


Check for unsorted payers (note: online dashboard includes unsorted)

In [9]:
print("Number of unsorted payers (all_Q5107_cleaned): " + str((all_Q5107_cleaned["PAYER_ID"]==12).sum()))

all_Q5107_cleaned_CONTAINS_PAYER_ID = all_Q5107_cleaned[
    all_Q5107_cleaned["PAYER_ID"] != 12
]

print("Number of unsorted payers (all_Q5107_cleaned_CONTAINS_PAYER_ID): " + str((all_Q5107_cleaned_CONTAINS_PAYER_ID["PAYER_ID"]==12).sum()))

all_Q5107_cleaned_CONTAINS_PAYER_ID = all_Q5107_cleaned # TODO: revert the revert

Number of unsorted payers (all_Q5107_cleaned): 37783
Number of unsorted payers (all_Q5107_cleaned_CONTAINS_PAYER_ID): 0


Check Missing

In [10]:
print(all_Q5107_cleaned_CONTAINS_PAYER_ID[["NEGOTIATED_DOLLAR","NEGOTIATED_PERCENTAGE","GROSS_CHARGE", "ESTIMATED_ALLOWED_AMOUNT"]].isna().all(axis=1).sum())
# Should be 978 --> 710 after cleaning payer ID

733


Drop Provider-Payer Rates Missing All Rates

In [11]:
all_Q5107_cleaned_HAS_RATE = all_Q5107_cleaned_CONTAINS_PAYER_ID[~all_Q5107_cleaned_CONTAINS_PAYER_ID[["NEGOTIATED_DOLLAR","NEGOTIATED_PERCENTAGE","GROSS_CHARGE", "ESTIMATED_ALLOWED_AMOUNT"]].isna().all(axis=1)]

len(all_Q5107_cleaned_HAS_RATE) # Should be 178534 --> 119491 after cleaning payer ID

127955

Remove Medicare

In [12]:
all_Q5107_cleaned_HAS_RATE = all_Q5107_cleaned_HAS_RATE[
    all_Q5107_cleaned_HAS_RATE["PAYER_CLASS_NAME"] == "Commercial"
]

Remove Inpatient

In [ ]:
print(len(all_Q5107_cleaned_HAS_RATE))

all_Q5107_cleaned_HAS_RATE = all_Q5107_cleaned_HAS_RATE[
    ~all_Q5107_cleaned_HAS_RATE["PLAN_NAME"].str.contains("inpatient", case = False, na = False)
]

print(len(all_Q5107_cleaned_HAS_RATE))

all_Q5107_cleaned_HAS_RATE = all_Q5107_cleaned_HAS_RATE[
    ~all_Q5107_cleaned_HAS_RATE["SETTING"].str.contains("Inpatient", case = False, na = False)
]

print(len(all_Q5107_cleaned_HAS_RATE))


63469
63426


Calculate Columns

In [13]:
all_Q5107_cleaned_HAS_RATE["MEDIAN_GROSS_CHARGE"] = (
    all_Q5107_cleaned_HAS_RATE.groupby(["PROVIDER_ID", "SETTING", "BILLING_CLASS"])["GROSS_CHARGE"].transform("median") # Imposes median for each provider, with variation for SETTING and BILLING_CLASS
)

all_Q5107_cleaned_HAS_RATE["FINAL_GROSS_CHARGE"] = all_Q5107_cleaned_HAS_RATE["MEDIAN_GROSS_CHARGE"] # Use this line to change which gross charge is used


all_Q5107_cleaned_HAS_RATE["NEGOTIATED_PERCENTAGE_TO_RATE"] = (
    all_Q5107_cleaned_HAS_RATE["NEGOTIATED_PERCENTAGE"] * all_Q5107_cleaned_HAS_RATE["FINAL_GROSS_CHARGE"] * 0.01
)

# Default to NEGOTIATED_DOLLAR, then NEGOTIATED_PERCENTAGE_TO_RATE (ESTIMATED_ALLOWED_AMOUNT ignored because has some implausible values)
all_Q5107_cleaned_HAS_RATE["BEST_RATE"] = (
    all_Q5107_cleaned_HAS_RATE[["NEGOTIATED_DOLLAR", "NEGOTIATED_PERCENTAGE_TO_RATE"]]
    .bfill(axis=1)
    .iloc[:, 0]
)

Add % Medicare

In [14]:
all_Q5107_cleaned_HAS_RATE["PERCENT_OF_MEDICARE"] = (
    (all_Q5107_cleaned_HAS_RATE["BEST_RATE"] / all_Q5107_cleaned_HAS_RATE["MEDICARE_RATE"]) * 100
)

Filter to 2025 List

In [15]:
providers_2025 = [
    1275797839,
    1174582050,
    1104166313,
    1851333686,
    1447595574,
    1679525919,
    1992703540,
    1588640692,
    1922459163,
    1700872355,
    1447359997,
    1609824010,
    1134144801,
    1952332801,
    1346294519,
    1811955917,
    1639172372,
    1548202641,
    1831251230,
    1740773936,
    1841439908,
    1932208576,
    1457458598,
    1447250253,
    1760590962,
    1730132515,
    1003961251,
    1649736752,
    1730130980,
    1124381959,
    1548419526,
    1871543215,
    1962511147,
    1811944101,
    1609863448,
    1730195017,
    1700887411,
    1235735432,
    1962876045,
    1043208457,
    1508956509,
    1487621496,
    1336139500,
    1902803315,
    1396882205,
    1649306754,
    1851458426,
    1487676433,
    1215097910,
    1205065018,
    1306938071
]

payers_2025 = [
    97,
    7,
    810,
    522,
    155,
    643,
    42,
    625,
    76,
    813,
    638,
    504,
    796,
    151,
    633,
    567,
    720,
    403,
    229,
    960,
    825,
    472,
    510,
    299,
    541,
    564,
    169,
    277,
    773,
    728,
    829,
    706,
    803,
    398,
    505,
    354,
    44,
    552,
    61,
    52,
    392,
    49,
    487,
    971,
    317,
    272,
    726,
    820,
    958,
    624,
    770,
    160,
    300,
    101,
    952,
    174,
    786,
    56,
    818,
    723
]

all_Q5107_cleaned_HAS_RATE= all_Q5107_cleaned_HAS_RATE[
    all_Q5107_cleaned_HAS_RATE["NPI"].isin(providers_2025) &
    all_Q5107_cleaned_HAS_RATE["PAYER_ID"].isin(payers_2025)
]

In [16]:
print(len(all_Q5107_cleaned_HAS_RATE))

294


Create Seperate Table For All Providers

In [17]:
print(all_Q5107_cleaned_HAS_RATE[["NEGOTIATED_DOLLAR", "NEGOTIATED_PERCENTAGE"]].notna().all(axis=1).sum())
print(all_Q5107_cleaned_HAS_RATE[["NEGOTIATED_DOLLAR", "NEGOTIATED_PERCENTAGE", "ESTIMATED_ALLOWED_AMOUNT"]].notna().all(axis=1).sum())

11
0


In [18]:
providers = (
    all_Q5107_cleaned_HAS_RATE
    .groupby("PROVIDER_ID", as_index = False)
    .agg(
        PROVIDER_NAME = ("PROVIDER_NAME", "first"),
        PROVIDER_ID =  ("PROVIDER_ID", "first"),
        NPI = ("NPI", "first"),
        HEALTH_SYSTEM_NAME = ("HEALTH_SYSTEM_NAME", "first"),
        STATE = ("STATE", "first"),
        STREET_ADDRESS = ("STREET_ADDRESS", "first"),
        MEDIAN_GROSS_CHARGE = ("GROSS_CHARGE", "median"), # TODO: Check if different
        MEDIAN_DISCOUNTED_CASH_RATE = ("DISCOUNTED_CASH_RATE", "median"), # TODO: Check if different
        MEDIAN_NEGOTIATED_DOLLAR = ("NEGOTIATED_DOLLAR", "median"),
        MEDIAN_NEGOTIATED_PERCENTAGE = ("NEGOTIATED_PERCENTAGE", "median"),
        MEDIAN_NEGOTIATED_PERCENTAGE_TO_RATE = ("NEGOTIATED_PERCENTAGE_TO_RATE", "median"),
        MEDIAN_ESTIMATED_ALLOWED_AMOUNT = ("ESTIMATED_ALLOWED_AMOUNT", "median"),
        MEDIAN_BEST_RATE = ("BEST_RATE", "median"),
        NUM_RATES = ("PROVIDER_ID", "size")


    )
)

print(providers[["MEDIAN_DISCOUNTED_CASH_RATE","MEDIAN_NEGOTIATED_DOLLAR","MEDIAN_NEGOTIATED_PERCENTAGE_TO_RATE", "MEDIAN_ESTIMATED_ALLOWED_AMOUNT", "MEDIAN_GROSS_CHARGE"]].isna().all(axis=1).sum()) # TODO: why >0?, probably because has one of negotiated percentage and gross charge but none of the other

0


In [19]:
print(len(providers)) # Should be 3063 --> to 3057 after dropping --> 2823 after cleaning provider ID

17


In [20]:
providers_online = [
    "CHRISTUS Santa Rosa Hospital - Medical Center",
    "IU Health University Hospital",
    "Adventist Health Castle",
    "DHR Health Womens Hospital",
    "PeaceHealth United General Medical Center",
    "Heart Hospital of Austin",
    "Baptist Health Floyd",
    "NewYork-Presbyterian Lower Manhattan Hospital",
    "Pottstown Hospital",
    "St Anthony Hospital"
]

providers_displayed_online= providers[
    providers["PROVIDER_NAME"].isin(providers_online)
]